In [1]:
# Libraries
import pandas as pd
import requests
from multiprocessing.pool import ThreadPool as Pool
from tqdm.notebook import tqdm
import json

In [ ]:
# Reading the data
data = pd.read_csv("data\Mastodon_instances_edges_Sept2024.csv")
data.head()

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\alelc\AppData\Local\Temp\ipykernel_28188\4284201516.py:1: SyntaxWarning: invalid escape sequence '\M'
  data = pd.read_csv("data\Mastodon_instances_edges_Sept2024.csv")


,source_domain,target_domain,weight
0,baraag.net,baraag.net,5048993
1,mastodon.social,mastodon.social,1156269
2,mastodon.uno,mastodon.uno,602899
3,mastodon.gamedev.place,mastodon.gamedev.place,344733
4,pawoo.net,baraag.net,305426


In [3]:
data.shape

(1734054, 3)

In [ ]:
# Removing self-loops
data = data[data['source_domain'] != data['target_domain']]
print(f"Dataset shape after removing self-loops: {data.shape}")

Dataset shape after removing self-loops: (1731393, 3)


In [5]:
data.shape

(1731393, 3)

In [ ]:
# Filtering the data
data = data[data['weight'] >= 10]
print(data.shape)

(217879, 3)


In [ ]:
# Saving the filtered data
data.to_csv("data/Mastodon_instances_edges_Sept2024_filtered.csv", index=False)

In [ ]:
import regex as re
from bs4 import BeautifulSoup

# Method for cleaning text
def clean_html(text):
    # Step 1: Parse HTML content
    soup = BeautifulSoup(text, "html.parser")

    # Step 2: Extract plain text from the parsed HTML (this also removes tags)
    clean_text = soup.get_text()

    # Step 3: Remove emojis using a regular expression
    # This regex allows all alphabets (any script), numbers, and spaces but removes emojis and symbols.
    clean_text = re.sub(r'[^\p{L}\p{N}\s]+', '', clean_text)  # \p{L} matches any letter, \p{N} matches any number

    # Step 4: Remove extra spaces, newlines, and tabs
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text

In [10]:
def get_instance_info(instance):
    # Preparing the target URL
    info_url = f'https://{instance}/api/v2/instance'

    # Asking for instance metadata
    try:
        # Performing the request
        response = requests.get(info_url, timeout=(19,10))

        # Checking for the status code
        if response.status_code == requests.codes.ok:
            # If ok get the json
            parsed_response = json.loads(response.text)
            parsed_rules = [rule['text'] for rule in parsed_response['rules']]

            active_users = parsed_response['usage']['users']['active_month']
            if active_users <= 0:
                return (instance, None, None)
            
            # Additional API call for extended description
            extended_desc_detailed = None
            try:
                # Preparing the target URL for extended description
                ext_desc_url = f'https://{instance}/api/v1/instance/extended_description'
                
                # Performing the request
                ext_response = requests.get(ext_desc_url, timeout=30)
                
                # Checking for the status code
                if ext_response.status_code == requests.codes.ok:
                    # If ok get the json
                    ext_parsed_response = json.loads(ext_response.text)
                    extended_desc_detailed = clean_html(ext_parsed_response['content'])
                    print(f"Extended description found for {instance}")
                
                # 404 means no ext. description was declared
                elif ext_response.status_code == 404:
                    print(f"{instance} didn't declare any extended description!")
                    extended_desc_detailed = None
                    
            except Exception as e:
                print(f"The following error occurred while asking for {instance}'s extended description: {e}")
                extended_desc_detailed = None
            
            return (instance, parsed_response['title'], clean_html(parsed_response['description']), extended_desc_detailed,
                    parsed_response['version'], parsed_response['source_url'], 
                    active_users, parsed_rules)

    except Exception as e:
        print(f"Error for instance: {instance}")
        print(e)
        return (instance, None, None)

In [11]:
# Test
get_instance_info("planetearth.social")

Extended description found for planetearth.social


('planetearth.social',
 'planetearth.social',
 'Join planetearthsocial Show Your Love for Earth on a GeneralPurpose Mastodon Server',
 'Join planetearthsocial A Growing Community for Earth Lovers and Beyond Are you passionate about our beautiful planet and eager to connect with likeminded individuals who share your love for Mother Earth We invite you to be part of something special Welcome to planetearthsocial a new and blossoming Mastodon server dedicated to fostering sustainability environmental consciousness and diverse discussions As a new server we are excited to build a thriving community from the ground up and we need your unique voice to help us grow By joining planetearthsocial youll have the opportunity to shape the conversations forge new connections and make a meaningful impact within our closeknit community Heres why you should join us Be a Founding Member As an early adopter youll have the chance to be part of our servers foundation Share your ideas suggestions and intere

In [12]:
# Split data into chunks
CHUNK_SIZE = 1000

def chunkify(data, size):
    for i in range(0, len(data), size):
        yield data[i:i+size]

In [13]:
# Parallelized execution
N_PROCESSES = 8
pool = Pool(N_PROCESSES)

# Results
results = []

# Chunks
chunks = list(chunkify(instances_list, CHUNK_SIZE))
print(f"Extracted {len(chunks)} chunks")

# Executing
for chunk in tqdm(chunks):
    pbar = tqdm(desc=f"Checking software",total=len(chunk))
    for res in pool.imap_unordered(get_instance_info, chunk):
        if res is not None:
            results.append(res)
        pbar.set_description(f"Checked(s): {len(results)} | Processed")
        pbar.update()
    pbar.close()

print("Instances checked: ", len(results))

Extracted 7 chunks


  0%|          | 0/7 [00:00<?, ?it/s]

Checking software:   0%|          | 0/1000 [00:00<?, ?it/s]

pleroma.lord.re didn't declare any extended description!
Extended description found for leipzig.town
Error for instance: succubus.town
HTTPSConnectionPool(host='succubus.town', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E231B4C560>: Failed to resolve 'succubus.town' ([Errno 11001] getaddrinfo failed)"))
Extended description found for v2br.social
Extended description found for social.bund.de
Extended description found for fosspri.de
Extended description found for mstdn.lt
Error for instance: social.hispabot.freemyip.com
HTTPSConnectionPool(host='social.hispabot.freemyip.com', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E231B4CC50>: Failed to resolve 'social.hispabot.freemyip.com' ([Errno 11001] getaddrinfo failed)"))
Extended description found for indg.club
Extended description found for

Checking software:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Error for instance: oarsport.social
HTTPSConnectionPool(host='oarsport.social', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E23246A210>: Failed to resolve 'oarsport.social' ([Errno 11001] getaddrinfo failed)"))
Error for instance: micro.inflo.ws
HTTPSConnectionPool(host='micro.inflo.ws', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E232469C40>: Failed to resolve 'micro.inflo.ws' ([Errno 11001] getaddrinfo failed)"))
Error for instance: humangenetics.social
HTTPSConnectionPool(host='humangenetics.social', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E231C84140>: Failed to resolve 'humangenetics.social' ([Errno 11001] getaddrinfo failed)"))
Extended description found for toot.plus.yt
Error fo

Checking software:   0%|          | 0/1000 [00:00<?, ?it/s]

Error for instance: stop.voring.me
HTTPSConnectionPool(host='stop.voring.me', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E2323C17C0>: Failed to resolve 'stop.voring.me' ([Errno 11001] getaddrinfo failed)"))
Error for instance: woofed.me
HTTPSConnectionPool(host='woofed.me', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E231A92A50>: Failed to resolve 'woofed.me' ([Errno 11001] getaddrinfo failed)"))
Error for instance: volksverpetzer.social
HTTPSConnectionPool(host='volksverpetzer.social', port=443): Max retries exceeded with url: /api/v2/instance (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1000)')))
Error for instance: law-and-politics.online
HTTPSConnectionPool(host='law-and-politics.online',

Checking software:   0%|          | 0/199 [00:00<?, ?it/s]

Extended description found for sbg-social.at
Extended description found for drg.nz
Extended description found for hostsharing.coop
Extended description found for corneill.es
Error for instance: screengeeks.club
HTTPSConnectionPool(host='screengeeks.club', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E232D94AD0>: Failed to resolve 'screengeeks.club' ([Errno 11001] getaddrinfo failed)"))
Error for instance: dancingbanana.party
HTTPSConnectionPool(host='dancingbanana.party', port=443): Max retries exceeded with url: /api/v2/instance (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E232DF52E0>: Failed to resolve 'dancingbanana.party' ([Errno 11001] getaddrinfo failed)"))
Error for instance: social.calidris.lv
HTTPSConnectionPool(host='social.calidris.lv', port=443): Max retries exceeded with url: /api/v2/instance (Caused by SSLError(SSLCertVerificationErr

In [14]:
# Creating dataframe from results
df_info = pd.DataFrame(results, columns=["instance", "title", "description", "extended_description", "version", "server_source", "active_users", "rules"])
df_info.head()

,instance,title,description,extended_description,version,server_source,active_users,rules
0,pleroma.lord.re,Pleroma,Monouser instance,None,2.7.2 (compatible; Pleroma 2.9.1),https://git.pleroma.social/pleroma/pleroma,2.0,[]
1,leipzig.town,leipzig.town,Hier sind alle Willkommen die Leipzig ihr zu H...,leipzigtown ist ein Angebot von Impressum Anga...,4.4.1,https://github.com/mastodon/mastodon,56.0,[Sexuell explizite oder gewalttätige Medien mü...
2,succubus.town,None,None,None,None,None,NaN,None
3,v2br.social,v2br.social,A place for vTubers their fans artists and the...,v2brsocial was created to be a place for vTube...,4.4.2,https://github.com/mastodon/mastodon,14.0,"[Don't be a jerk., No racism, sexism, homophob..."
4,social.bund.de,social.bund.de,Dies ist der MastodonServer der Bundesbeauftra...,Die Bundesbeauftragte für den Datenschutz und ...,4.3.8,https://github.com/mastodon/mastodon,98.0,[]


In [15]:
df_info.shape

(5112, 8)

In [ ]:
# Removing rows with missing values
df_info_clean = df_info.dropna()
df_info_clean.head()

,instance,title,description,extended_description,version,server_source,active_users,rules
1,leipzig.town,leipzig.town,Hier sind alle Willkommen die Leipzig ihr zu H...,leipzigtown ist ein Angebot von Impressum Anga...,4.4.1,https://github.com/mastodon/mastodon,56.0,[Sexuell explizite oder gewalttätige Medien mü...
3,v2br.social,v2br.social,A place for vTubers their fans artists and the...,v2brsocial was created to be a place for vTube...,4.4.2,https://github.com/mastodon/mastodon,14.0,"[Don't be a jerk., No racism, sexism, homophob..."
4,social.bund.de,social.bund.de,Dies ist der MastodonServer der Bundesbeauftra...,Die Bundesbeauftragte für den Datenschutz und ...,4.3.8,https://github.com/mastodon/mastodon,98.0,[]
5,fosspri.de,FossPride™,,This instance uses Mutant Standard emoji which...,4.4.2+fosspride,https://github.com/Joshix-1/mastodon/tree/v4.4...,7.0,"[be nice, don't toot illegal things, show resp..."
6,mstdn.lt,Visos upės teka,Čia suprantam lietuviškai We understand lithua...,Our small Mastodon server is managed by lithua...,4.4.2+glitch,https://github.com/glitch-soc/mastodon,41.0,[Be respectful and courteous to others at all ...


In [18]:
df_info_clean.isna().sum()

instance                0
title                   0
description             0
extended_description    0
version                 0
server_source           0
active_users            0
rules                   0
dtype: int64

In [19]:
df_info_clean.shape

(3096, 8)

In [20]:
import ast

# Convert 'rules' column from string representation to actual list of dictionaries if needed
def parse_rules(rules):
    if isinstance(rules, str):
        try:
            return ast.literal_eval(rules)
        except Exception:
            return []
    return rules

df_info_clean['rules'] = df_info_clean['rules'].apply(parse_rules)

C:\Users\alelc\AppData\Local\Temp\ipykernel_28188\1250915123.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_info_clean['rules'] = df_info_clean['rules'].apply(parse_rules)


In [21]:
df_info_clean.head()

,instance,title,description,extended_description,version,server_source,active_users,rules
1,leipzig.town,leipzig.town,Hier sind alle Willkommen die Leipzig ihr zu H...,leipzigtown ist ein Angebot von Impressum Anga...,4.4.1,https://github.com/mastodon/mastodon,56.0,[Sexuell explizite oder gewalttätige Medien mü...
3,v2br.social,v2br.social,A place for vTubers their fans artists and the...,v2brsocial was created to be a place for vTube...,4.4.2,https://github.com/mastodon/mastodon,14.0,"[Don't be a jerk., No racism, sexism, homophob..."
4,social.bund.de,social.bund.de,Dies ist der MastodonServer der Bundesbeauftra...,Die Bundesbeauftragte für den Datenschutz und ...,4.3.8,https://github.com/mastodon/mastodon,98.0,[]
5,fosspri.de,FossPride™,,This instance uses Mutant Standard emoji which...,4.4.2+fosspride,https://github.com/Joshix-1/mastodon/tree/v4.4...,7.0,"[be nice, don't toot illegal things, show resp..."
6,mstdn.lt,Visos upės teka,Čia suprantam lietuviškai We understand lithua...,Our small Mastodon server is managed by lithua...,4.4.2+glitch,https://github.com/glitch-soc/mastodon,41.0,[Be respectful and courteous to others at all ...


In [22]:
# Remove rows with no description
df_info_with_description = df_info_clean[df_info_clean['description'].notna() & (df_info_clean['description'] != '')]
print(f"Original dataframe shape: {df_info_clean.shape}")
print(f"After removing rows with no description: {df_info_with_description.shape}")
print(f"Removed {df_info_clean.shape[0] - df_info_with_description.shape[0]} rows with missing descriptions")

# Show some examples of remaining descriptions
print("\nSample descriptions:")
print(df_info_with_description['description'].head().tolist())

Original dataframe shape: (3096, 8)
After removing rows with no description: (2630, 8)
Removed 466 rows with missing descriptions

Sample descriptions:
['Hier sind alle Willkommen die Leipzig ihr zu Hause nennen undoder die Stadt lieben Bitte seid exzellent zueinander und haltet euch an die Regeln Foto von F Heiberger Pixabay', 'A place for vTubers their fans artists and the everyone else to gather share and have fun', 'Dies ist der MastodonServer der Bundesbeauftragten für den Datenschutz und die Informationsfreiheit BfDI', 'Čia suprantam lietuviškai We understand lithuanian', 'Indigenous People Hosted server for Indigenous People Culture Language banner image Nanibah']


In [ ]:
# Saving the cleaned dataset with metadata
df_info_with_description.to_csv("data/Mastodon_instances_metadata_v3.csv", index=False)